# Prediksi Stroke - Machine Learning Model

Notebook ini akan membangun model machine learning untuk memprediksi risiko stroke berdasarkan data kesehatan.

## Outline:
1. Import Required Libraries
2. Load and Explore Dataset Stroke
3. Data Preprocessing and Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Split Data into Training and Testing Sets
7. Build and Train Machine Learning Model
8. Model Evaluation and Validation
9. Make Predictions on New Data
10. Save the Trained Model

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set style untuk visualisasi
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load and Explore Dataset Stroke

In [ ]:
# Load dataset
df = pd.read_csv('../dataset_stroke/healthcare-dataset-stroke-data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

## 3. Data Preprocessing and Cleaning

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nPercentage of Missing Values:")
print((df.isnull().sum() / len(df) * 100).round(2))

# Handle missing values in BMI
# Replace 'N/A' with NaN dan hitung median BMI
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Drop id column (tidak diperlukan untuk prediksi)
df = df.drop('id', axis=1)

# Check class distribution untuk stroke (target variable)
print("\nTarget Variable Distribution (Stroke):")
print(df['stroke'].value_counts())
print("\nTarget Variable Percentage:")
print(df['stroke'].value_counts(normalize=True) * 100)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Visualisasi distribusi fitur numerik
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Age distribution
axes[0, 0].hist(df['age'], bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Glucose level distribution
axes[0, 1].hist(df['avg_glucose_level'], bins=30, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Glucose Level Distribution')
axes[0, 1].set_xlabel('Glucose Level')
axes[0, 1].set_ylabel('Frequency')

# BMI distribution
axes[1, 0].hist(df['bmi'], bins=30, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('BMI Distribution')
axes[1, 0].set_xlabel('BMI')
axes[1, 0].set_ylabel('Frequency')

# Stroke distribution
stroke_counts = df['stroke'].value_counts()
axes[1, 1].bar(['No Stroke', 'Stroke'], stroke_counts.values, color=['green', 'red'])
axes[1, 1].set_title('Stroke Distribution')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Categorical features distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].value_counts(df['gender']).plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Gender Distribution')
axes[0, 0].set_ylabel('Count')

axes[0, 1].value_counts(df['ever_married']).plot(kind='bar', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Ever Married Distribution')
axes[0, 1].set_ylabel('Count')

axes[0, 2].value_counts(df['work_type']).plot(kind='bar', ax=axes[0, 2], color='lightgreen')
axes[0, 2].set_title('Work Type Distribution')
axes[0, 2].set_ylabel('Count')

axes[1, 0].value_counts(df['Residence_type']).plot(kind='bar', ax=axes[1, 0], color='lightyellow')
axes[1, 0].set_title('Residence Type Distribution')
axes[1, 0].set_ylabel('Count')

axes[1, 1].value_counts(df['smoking_status']).plot(kind='bar', ax=axes[1, 1], color='lightblue')
axes[1, 1].set_title('Smoking Status Distribution')
axes[1, 1].set_ylabel('Count')

axes[1, 2].value_counts(df['hypertension']).plot(kind='bar', ax=axes[1, 2], color='lightsalmon')
axes[1, 2].set_title('Hypertension Distribution')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Membuat copy untuk preprocessing
df_processed = df.copy()

# Encode categorical variables menggunakan LabelEncoder
label_encoders = {}
categorical_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for feature in categorical_features:
    le = LabelEncoder()
    df_processed[feature] = le.fit_transform(df_processed[feature])
    label_encoders[feature] = le

# Pisahkan features dan target
X = df_processed.drop('stroke', axis=1)
y = df_processed['stroke']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:")
print(X.columns.tolist())
print("\nData after encoding:")
print(X.head())

## 6. Split Data into Training and Testing Sets

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nTraining set stroke distribution:")
print(y_train.value_counts())
print(f"\nTesting set stroke distribution:")
print(y_test.value_counts())

# Scale the features menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaling completed!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

## 7. Build and Train Machine Learning Model

In [ ]:
# Opsi 1: Logistic Regression
print("Training Logistic Regression Model...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Make predictions
lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

print("Logistic Regression model trained!")

# Opsi 2: Random Forest (menggunakan model ini karena lebih robust)
print("\nTraining Random Forest Model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

print("Random Forest model trained!")

# Kita akan menggunakan Random Forest sebagai model utama
model = rf_model
model_name = "Random Forest"
X_test_pred = rf_test_pred
X_train_pred = rf_train_pred

## 8. Model Evaluation and Validation

In [ ]:
# Evaluate model performance
def evaluate_model(model, X_train, X_test, y_train, y_test, y_train_pred, y_test_pred, model_name):
    print(f"\n{'='*60}")
    print(f"{model_name} - Model Evaluation")
    print(f"{'='*60}")
    
    # Training metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred)
    train_recall = recall_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred)
    
    print(f"\nTraining Metrics:")
    print(f"  Accuracy:  {train_accuracy:.4f}")
    print(f"  Precision: {train_precision:.4f}")
    print(f"  Recall:    {train_recall:.4f}")
    print(f"  F1-Score:  {train_f1:.4f}")
    
    # Testing metrics
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    print(f"\nTesting Metrics:")
    print(f"  Accuracy:  {test_accuracy:.4f}")
    print(f"  Precision: {test_precision:.4f}")
    print(f"  Recall:    {test_recall:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")
    
    # ROC-AUC Score
    test_proba = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, test_proba)
    print(f"\nROC-AUC Score: {roc_auc:.4f}")
    
    # Classification Report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_test_pred, target_names=['No Stroke', 'Stroke']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    # Plot confusion matrix
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    return test_accuracy

# Evaluate Random Forest model
accuracy = evaluate_model(model, X_train, X_test, y_train, y_test, X_train_pred, X_test_pred, model_name)

## 9. Make Predictions on New Data

In [ ]:
# Example: Make prediction on new data
# Sample data untuk testing
sample_data = {
    'gender': 'Male',
    'age': 67,
    'hypertension': 0,
    'heart_disease': 1,
    'ever_married': 'Yes',
    'work_type': 'Private',
    'Residence_type': 'Urban',
    'avg_glucose_level': 228.69,
    'bmi': 36.6,
    'smoking_status': 'formerly smoked'
}

# Encode sample data
sample_encoded = []
for feature in X.columns:
    if feature in categorical_features:
        le = label_encoders[feature]
        sample_encoded.append(le.transform([sample_data[feature]])[0])
    else:
        sample_encoded.append(sample_data[feature])

sample_array = np.array([sample_encoded])

# Make prediction
prediction = model.predict(sample_array)[0]
probability = model.predict_proba(sample_array)[0][1]

print("Sample Input Data:")
print(sample_data)
print(f"\nPrediction: {'Stroke Risk' if prediction == 1 else 'No Stroke Risk'}")
print(f"Probability: {probability:.4f} ({probability*100:.2f}%)")

# Determine risk level berdasarkan probability
if probability >= 0.70:
    risk_level = "Tinggi"
elif probability >= 0.40:
    risk_level = "Sedang"
else:
    risk_level = "Rendah"

print(f"Risk Level: {risk_level}")

## 10. Save the Trained Model

In [ ]:
# Save model ke format pickle
model_path = '../model/stroke_model.pkl'
scaler_path = '../model/stroke_scaler.pkl'

with open(model_path, 'wb') as file:
    pickle.dump(model, file)
    print(f"Model saved to {model_path}")

with open(scaler_path, 'wb') as file:
    pickle.dump(scaler, file)
    print(f"Scaler saved to {scaler_path}")

# Save label encoders
encoder_path = '../model/stroke_label_encoders.pkl'
with open(encoder_path, 'wb') as file:
    pickle.dump(label_encoders, file)
    print(f"Label Encoders saved to {encoder_path}")

print("\n✓ All models saved successfully!")
print(f"\nModel Information:")
print(f"- Model Type: Random Forest")
print(f"- Training Accuracy: {accuracy_score(y_train, X_train_pred):.4f}")
print(f"- Testing Accuracy: {accuracy:.4f}")
print(f"- Number of Features: {X.shape[1]}")
print(f"- Feature Names: {X.columns.tolist()}")